# 28 · Prototype v2: kiểm lại đường giao diện (R-OCR v2 · R-LIVE v2) + ví dụ demo + quay video (W14 bước 3 + 4)

**Vấn đề (W13 bước 8 + W14 bước 1–2):**
- R-OCR **không đạt** ở W13 (ViTextVQA 87,5%). Nguyên nhân chính: giao diện thu nhỏ ảnh lớn hơn 2.048 px, trong khi **1.418 ảnh ViTextVQA** lớn hơn ngưỡng đó. Ghi chú W13 "ảnh dataset đều nhỏ hơn" là sai.
- OCR cache ViTextVQA có lỗi EXIF (đã sửa ở W14 bước 2). Cache đúng để so là `paddledet_vietocr_exiffix.json`.
- Video demo dự phòng chưa quay.

**Bản sửa (`prototype-app-v2`):** giao diện chỉ thu nhỏ ảnh lớn hơn **6.000 px**, nên mọi ảnh dataset (lớn nhất 5.770 px) đi **nguyên file** như lúc eval. Ảnh có cờ xoay EXIF vẫn được xoay. `ocr.py` (W14) tự xoay nếu nhận ảnh còn cờ.

**Luật ghi trước** (`docs/W14/README.md`, mục "Bước 3"):

| Luật | ĐẠT khi |
|---|---|
| **R-OCR v2** | văn bản OCR vào prompt **qua giao diện** trùng hệt OCR **đọc thẳng file gốc** bằng cùng tiến trình OCR, cùng phiên L4, ở ≥ 98% ảnh, từng tập. So với cache (dựng trên T4) chỉ **mô tả** |
| **R-LIVE v2** | như W13: câu trả lời qua giao diện trùng run eval W11 ở ≥ 97% câu **và** EM không thấp hơn quá 1,5 điểm, từng tập. Bỏ câu có/không, và **bỏ thêm** câu của ảnh có cờ xoay EXIF |

**Notebook này:** cài thư viện → kiểm → chạy 200 ảnh / tập qua đường giao diện + OCR đọc thẳng → phán quyết → ảnh thật → chọn ví dụ demo bằng luật cố định → **khối gửi Claude** → mở giao diện **30 phút** để quay video → tự trả máy.

**Runtime:** **L4**. Khoảng 75 phút (≈ 2 CU), trong đó 30 phút là giao diện mở để quay video.

**Anh chỉ cần:** Run all, gửi Claude **output mục 7**, rồi quay video theo `docs/W14/Demo_Script.md` khi link giao diện hiện ra (mục 8).

## 1. Thiết lập môi trường

*Kỳ vọng: commit mới nhất (W14 bước 3–4), `Runtime: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, hashlib, subprocess, glob
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
from src.eval.prototype_live import LIVE_VERSION, LIVE_RUN, OUT_NAME, REAL_RUN, DIRECT_OCR_FILE
from src.prototype.app import APP_VERSION
assert LIVE_VERSION == 'prototype-live-v2' and APP_VERSION.startswith('prototype-app-v2'), 'Code cũ -> git pull chưa lấy được commit W14 bước 3'
assert 'Bước 3' in open('docs/W14/README.md', encoding='utf-8').read(), '❌ Luật bước 3 chưa được commit -> DỪNG'

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
OUT = f'{EXP_ROOT}/{OUT_NAME}'
PHOTOS = f'{DRIVE_ROOT}/data/real_photos_w13'          # ảnh thật của nhóm + cau_hoi.csv (tuỳ chọn)
UI_MINUTES = 30                                        # thời gian mở giao diện để quay video (phút); 0 = không mở

def run_cmd(cmd, check=True):
    # Chạy lệnh, in log trực tiếp; lỗi -> DỪNG notebook (trừ khi check=False: trả mã lỗi)
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {rc}) -> xem log ở trên, gửi Claude')
    return rc

GPU = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip() or 'CPU'
print('Runtime:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Notebook 28 phải chạy trên L4 (cùng GPU với run eval W11). '
                     'Runtime -> Change runtime type -> L4, rồi Run all.')
print('✅ Thiết lập xong')

## 2. Cài thư viện cho Qwen **và** OCR trong cùng phiên (≈ 6–8 phút)

**Vấn đề (W08, notebook 10/19):** `paddlepaddle-gpu` bản CUDA 12.6 cài kèm NCCL của CUDA 12, **ghi đè** file NCCL mà torch của Colab đang dùng, làm `import torch` hỏng.

**Cell này làm theo đúng quy trình đã chạy được ở notebook 19:**
1. Cài thư viện của Qwen (cùng pin với notebook 23/24: `transformers==5.17.0`), Gradio (`6.31.0`), VietOCR (`--no-deps`).
2. Chụp danh sách thư viện `nvidia-*` của torch → cài `paddleocr==3.7.0` + `paddlepaddle-gpu==3.3.1` (cùng phiên bản đã dựng cache; thử tối đa 3 lần) → cài lại NCCL của torch.
3. Kiểm ở tiến trình riêng: torch thấy GPU **và** Paddle chạy GPU → `OCR_DEVICE = gpu`. Không được thì gỡ Paddle GPU, khôi phục torch, cài Paddle CPU → `OCR_DEVICE = cpu` (R-OCR sẽ đo xem OCR CPU có còn trùng cache không).
4. Kiểm lần cuối: torch + transformers 5.17 + peft + bitsandbytes còn tốt.

*Kỳ vọng: `torch OK trước khi cài`, `paddleocr 3.7.0 | paddle 3.3.1`, `OCR_DEVICE = gpu` (hoặc `cpu` kèm lý do), `✅ torch + transformers 5.17.0 vẫn tốt`.*

In [ ]:
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-1500:]

def nvidia_pkgs():
    out = subprocess.run('pip list --format=json', shell=True, capture_output=True, text=True).stdout
    return {p['name'].lower(): p['version'] for p in json.loads(out) if p['name'].lower().startswith('nvidia-')}

def torch_ok():
    return sh('python -c "import torch; assert torch.cuda.is_available(); torch.zeros(1).cuda()"')[0] == 0

def paddle_gpu_ok():
    return sh('python -c "import paddle; assert paddle.device.is_compiled_with_cuda(); paddle.utils.run_check()"')[0] == 0

def restore(pkgs):                       # cài lại ĐÚNG phiên bản cũ, không kéo thêm gì
    if pkgs:
        code_, log = sh('pip install -q --force-reinstall --no-deps ' + ' '.join(f'{n}=={v}' for n, v in pkgs.items()))
        assert code_ == 0, log

assert torch_ok(), '❌ torch đã hỏng từ trước -> Runtime -> Disconnect and delete runtime, rồi Run all lại'
print('torch OK trước khi cài')
code_, log = sh('pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml gradio==6.31.0')
assert code_ == 0, log
code_, log = sh('pip install -q --no-deps vietocr==0.3.13 && pip install -q gdown einops')
assert code_ == 0, log
code_, log = sh('pip install -q paddleocr==3.7.0')
assert code_ == 0, log
BEFORE = nvidia_pkgs()

for attempt in range(1, 4):
    code_gpu, log_gpu = sh('pip install -q --timeout 180 --retries 5 paddlepaddle-gpu==3.3.1 '
                           '-i https://www.paddlepaddle.org.cn/packages/stable/cu126/')
    if code_gpu == 0:
        break
    print(f'⚠ Lần {attempt}/3: cài Paddle GPU lỗi -> thử lại. Đuôi log:', log_gpu[-200:].replace(chr(10), ' '))
NEW = sorted(set(nvidia_pkgs()) - set(BEFORE))
restore({n: v for n, v in BEFORE.items() if 'nccl' in n})
if not torch_ok():
    print('torch vẫn hỏng sau khi khôi phục NCCL -> khôi phục toàn bộ thư viện nvidia-* của torch')
    restore(BEFORE)

if code_gpu == 0 and torch_ok() and paddle_gpu_ok():
    OCR_DEVICE = 'gpu'
else:
    print('⚠ Paddle GPU không chạy chung được với torch' if code_gpu == 0 else '⚠ Cài Paddle GPU lỗi:\n' + log_gpu[-400:])
    print('-> gỡ Paddle GPU, khôi phục thư viện torch, cài Paddle CPU')
    sh('pip uninstall -y -q paddlepaddle-gpu ' + ' '.join(NEW))
    restore(BEFORE)
    code_, log = sh('pip install -q paddlepaddle')
    assert code_ == 0, log
    OCR_DEVICE = 'cpu'

assert torch_ok(), '❌ torch hỏng sau khi cài Paddle -> gửi toàn bộ output cell này cho Claude'
rc, out = sh('python -c "import transformers, peft, bitsandbytes, qwen_vl_utils, gradio; '
             'print(transformers.__version__, gradio.__version__)"')
assert rc == 0 and out.strip().startswith('5.17.0'), f'❌ thư viện Qwen hỏng sau khi cài Paddle:\n{out}'
print(sh('python -c "import paddleocr, paddle; print(\'paddleocr\', paddleocr.__version__, \'| paddle\', paddle.__version__)"')[1])
print('OCR_DEVICE =', OCR_DEVICE, '| ✅ torch + transformers', out.split()[0], 'vẫn tốt | gradio', out.split()[1])

## 3. Kiểm tra trước khi chạy (≈ 2 phút)

- **(a)** Selftest 6 script: `ocr_worker`, `app` (có Gradio → kiểm cả dựng giao diện), `prototype_live`, `prototype_check`, `demo_examples`, `ocr`.
- **(b)** Ba luật bước 4–6 của W13 đều ĐẠT.
- **(c)** Run eval W11 của SH có trên Drive; adapter SH **trùng** dấu vân tay run W11.
- **(d)** OCR cache: ViVQA như W13; ViTextVQA = **cache đã sửa EXIF**, trùng sha256 biên bản W14 bước 2.
- **(e)** Khởi động OCR thật ở tiến trình riêng: engine trùng engine đã dựng cache **và** có khoá `exif` (bản sửa W14); đọc thử 1 ảnh.

*Kỳ vọng: `✅ selftest OK (6 script)`, `luật bước 4–6: …ĐẠT…`, 2 dòng `mốc W11 … ✅`, `adapter SH = run W11 ✅`, `OCR cache vivqa ✅`, `OCR cache vitextvqa (sửa EXIF) ✅`, `engine OCR = cache ✅`, `exif: transpose-v1 (W14 R-EXIF)`.*

In [ ]:
# (a) selftest
musts = {'src.prototype.ocr_worker': ['selftest ocr_worker OK'],
         'src.prototype.app': ['selftest app OK', 'dựng Gradio'],
         'src.eval.prototype_live': ['selftest prototype_live OK'],
         'src.eval.prototype_check': ['prototype_check selftest OK'],
         'src.prototype.demo_examples': ['selftest demo_examples OK'],
         'src.data.ocr': ['ocr exif fix OK', 'ocr SELFTEST OK']}
for mod, need in musts.items():
    o = subprocess.run(f'python -m {mod} --selftest', shell=True, capture_output=True, text=True)
    for m in need:
        assert m in o.stdout, (mod, m, o.stdout[-800:], o.stderr[-800:])
print('✅ selftest OK (6 script)')

# (b) luật bước 4–6
from src.prototype.app import load_ui_settings, require_verified, load_ocr_selection
ST = load_ui_settings()
require_verified(ST)
print('luật bước 4–6:', ST['verdicts'])

# (c) mốc W11 + adapter
from src.eval.prototype_check import REF_RUN
from src.eval.run_baseline import adapter_fingerprint
for ds, run in REF_RUN.items():
    assert os.path.exists(f'{EXP_ROOT}/{run}/predictions.json'), f'thiếu {run}/predictions.json trên Drive -> báo Claude'
    print(f'mốc W11 {ds}: {run} ✅')
ref = json.load(open(f"{EXP_ROOT}/{REF_RUN['vivqa']}/metrics.json", encoding='utf-8'))['meta']['adapter']
AD_SH, AD_SHA = ref['path'], ref['weights_sha256']
assert adapter_fingerprint(AD_SH)['weights_sha256'] == AD_SHA, 'adapter SH trên Drive KHÁC run W11 -> dừng, báo Claude'
print(f'adapter SH = run W11 ✅ ({AD_SHA[:12]}…)')

# (d) OCR cache
from src.data.ocr import default_cache_path, load_ocr_cache, lines_to_texts
SEL = load_ocr_selection()
p = default_cache_path(DRIVE_ROOT, 'vivqa', SEL['backend'])
assert os.path.exists(p), f'thiếu OCR cache {p}'
print('OCR cache vivqa ✅')
from src.data.exif_audit import sha256_file
FIXMAN = json.load(open('experiments/W14_exif_fix/ocr_cache_exiffix_manifest.json', encoding='utf-8'))
p = default_cache_path(DRIVE_ROOT, 'vitextvqa_official', SEL['backend']).replace('.json', '_exiffix.json')
assert FIXMAN['valid'] and os.path.exists(p) and sha256_file(p) == FIXMAN['sha256'], f'❌ cache ViTextVQA đã sửa thiếu / khác biên bản: {p}'
print('OCR cache vitextvqa (sửa EXIF) ✅', FIXMAN['sha256'][:12])

# (e) OCR thật ở tiến trình riêng
from src.prototype.ocr_worker import OCRWorkerClient, check_engine_matches_cache
os.makedirs(OUT, exist_ok=True)
with OCRWorkerClient(backend=SEL['backend'], device=OCR_DEVICE, log_path=f'{OUT}/ocr_smoke.log') as c:
    chk = check_engine_matches_cache(c.info)
    print('engine OCR:', {k: c.info.get(k) for k in ('paddleocr', 'paddlepaddle', 'device', 'det_model', 'rec_model')})
    if chk['device_note']:
        print('ghi chú:', chk['device_note'])
    assert chk['ok'], f"❌ engine OCR khác engine đã dựng cache: {chk['diff']} -> gửi Claude output cell này"
    print('engine OCR = cache ✅')
    assert c.info.get('exif') == 'transpose-v1 (W14 R-EXIF)', f"❌ tiến trình OCR chưa có bản sửa EXIF: {c.info.get('exif')}"
    print('exif:', c.info['exif'])
    cache = load_ocr_cache(default_cache_path(DRIVE_ROOT, 'vivqa', SEL['backend']))
    img = next(k for k, v in cache['images'].items() if v.get('lines'))
    src_img = glob.glob(f'{DRIVE_ROOT}/data/vivqa/images/{img}')
    if src_img:
        live = lines_to_texts({'lines': c.lines(src_img[0])}, SEL['max_chars'], SEL['min_conf'])[0]
        ref_t = lines_to_texts(cache['images'][img], SEL['max_chars'], SEL['min_conf'])[0]
        print(f'đọc thử {img}: {len(live)} dòng chữ | trùng cache: {live == ref_t}')

## 4. Chạy đường giao diện trên 200 ảnh test mỗi tập + OCR đọc thẳng (≈ 25–35 phút)

`src.eval.prototype_live --run --version v2`:
- **Cùng mẫu** với W13 (200 ảnh / tập, seed 42, mọi câu hỏi của các ảnh đó).
- Mỗi câu: `prepare_image` (ngưỡng 6.000 px) → `Backend.ask` (OCR trực tiếp → prompt → Qwen + adapter SH).
- Sau đó, cùng tiến trình OCR **đọc thẳng file gốc** của từng ảnh → `direct_ocr.json` (mốc của luật R-OCR v2).
- Ghi lên Drive liên tục. **Bị ngắt thì Run all lại**: phần đã có tự bỏ qua.

*Kỳ vọng: `[live] mẫu: …`, `[live] khởi động … "ok": true …`, các dòng `[live] 50/…`, `[live] OCR đọc thẳng file gốc: còn 400 ảnh`, `[live] OCR thẳng 400/400`, `== xong … câu`.*

In [ ]:
if os.path.exists(f'{EXP_ROOT}/{LIVE_RUN}/meta.json') and os.path.exists(f'{EXP_ROOT}/{LIVE_RUN}/{DIRECT_OCR_FILE}'):
    print('⏭ đã chạy xong trước đó -> bỏ qua')
else:
    run_cmd(f'python -u -m src.eval.prototype_live --run --version v2 --drive_root {DRIVE_ROOT} --adapter "{AD_SH}" '
            f'--expect_adapter_sha {AD_SHA} --ocr_device {OCR_DEVICE}')

## 5. Phân tích + phán quyết (R-OCR v2 · R-LIVE v2) + tốc độ

*Kỳ vọng: bảng so với cache (chỉ mô tả, có cột độ giống), bảng "Giao diện = đọc thẳng", 2 dòng `➜ **R-OCR: …**`, `➜ **R-LIVE: …**`, bảng latency. Luật KHÔNG ĐẠT thì notebook **vẫn chạy tiếp**.*

In [ ]:
run_cmd(f'python -m src.eval.prototype_live --analyze --version v2 --drive_root {DRIVE_ROOT} --out {OUT}')

## 6. Ảnh thật của nhóm (tuỳ chọn, ≈ 2 phút)

Chạy nếu trên Drive có thư mục `ViVQA-VLM/data/real_photos_w13/` chứa ảnh **và** file `cau_hoi.csv` (mẫu: `docs/W13/realphotos_cau_hoi_mau.csv`; cột `anh`, `cau_hoi`, `dap_an` — đáp án có thể bỏ trống, nhiều cách viết thì tách bằng `|`). Không có thì cell tự bỏ qua.

Ảnh **không** được đưa lên GitHub; kết quả (câu hỏi, câu trả lời, mức, thời gian) ghi vào `experiments/W14_REAL_sh_realphotos_v2_bf16_L4/` trên Drive.

*Kỳ vọng: bảng `# W14 bước 3 — ảnh thật của nhóm`, hoặc `⏭ chưa có ảnh thật -> bỏ qua`.*

In [ ]:
CSV = f'{PHOTOS}/cau_hoi.csv'
if os.path.exists(CSV):
    run_cmd(f'python -u -m src.eval.prototype_live --realphotos --version v2 --photos_dir "{PHOTOS}" --questions_csv "{CSV}" '
            f'--drive_root {DRIVE_ROOT} --adapter "{AD_SH}" --expect_adapter_sha {AD_SHA} --ocr_device {OCR_DEVICE}',
            check=False)
else:
    print(f'⏭ chưa có ảnh thật ({CSV}) -> bỏ qua')

## 6b. Chọn ví dụ demo bằng luật cố định (`src/prototype/demo_examples.py`)

Ví dụ **không chọn tay**. Từ kết quả mục 4 (thứ tự mẫu seed 42), lấy câu đầu tiên thoả từng ô: đọc chữ đúng mức Cao (2), ảnh xoay EXIF, có/không, đếm, màu sắc, một câu **sai** mức Thấp. Sau đó thêm ảnh thật của nhóm. Bảng này dùng khi quay video (`docs/W14/Demo_Script.md`).

*Kỳ vọng: bảng `# W14 bước 4 — ví dụ demo …` khoảng 7–8 dòng dataset, cộng các dòng ảnh thật.*

In [ ]:
from src.eval.run_baseline import prepare_images
ROOTS = {ds: prepare_images(ds, DRIVE_ROOT) for ds in ('vivqa', 'vitextvqa_official')}
DEMO = f'{OUT}/demo'
real_args = f'--real_csv "{PHOTOS}/cau_hoi.csv" --real_dir "{PHOTOS}"' if os.path.exists(f'{PHOTOS}/cau_hoi.csv') else ''
run_cmd(f"python -m src.prototype.demo_examples --records {EXP_ROOT}/{LIVE_RUN}/records.jsonl "
        f"--image_roots vivqa={ROOTS['vivqa']} vitextvqa_official={ROOTS['vitextvqa_official']} {real_args} --out {DEMO}")

## 7. ⭐ Khối gửi Claude (gửi trước khi quay video)

*Kỳ vọng: khoảng 25 dòng, bắt đầu bằng `=== W14 bước 3 ===`.*

In [ ]:
print('=== W14 bước 3 ===')
!git log -1 --format='commit %h'
S = json.load(open(f'{OUT}/prototype_live_summary.json', encoding='utf-8'))
print('verdicts', S['verdicts'], '| GPU', S['meta']['gpu'], '| OCR', S['meta']['ocr_device'], '| max_side', S['rules']['max_side'])
for ds in ('vivqa', 'vitextvqa_official'):
    o, p, a = S['ocr'][ds], S['ocr_pipeline'][ds], S['answers'][ds]
    print(f"{ds}: giao diện=đọc thẳng {p['identical']:.4f} ({p['n_diff']} khác) | =cache {o['identical']:.4f} "
          f"({o['n_diff']} khác, giống≥0.9 {o.get('similar', 1):.4f}, min {o.get('min_similarity', 1):.3f}) | "
          f"R-LIVE trùng {a['identical']:.4f} n={a['n']} (bỏ có/không {a['skipped_yesno']}, bỏ ảnh xoay {a['skipped_rotated']}) "
          f"EM {a['em_live']:.2f} vs W11 {a['em_w11']:.2f}")
    for x in p['diffs'][:3]:
        print('   pipeline diff', x['image'], x['similarity'], x['live'][:3], '|', (x['cache'] or [])[:3])
    for x in a['mismatches'][:3]:
        print('   answer diff', x)
L = S['latency']['all']
print('latency all p50/p95 total', L['sec_total']['p50'], L['sec_total']['p95'], '| ocr', L['sec_ocr']['p50'], L['sec_ocr']['p95'],
      '| vlm', L['sec_vlm']['p50'], L['sec_vlm']['p95'])
print('startup', S['meta']['startup'].get('sec_start_total'), 's | VRAM main', S['meta']['peak_vram_gb_main'], 'GB | end', S['meta']['gpu_memory_end'].get('total_mib'), 'MiB')
print('ảnh xoay', S.get('n_rotated_records'), 'câu | thu nhỏ', S.get('n_resized_records'), 'câu | levels', S['levels'])
R = f'{EXP_ROOT}/{REAL_RUN}/realphotos_summary.json'
if os.path.exists(R):
    rr = json.load(open(R, encoding='utf-8'))['records']
    g = [r for r in rr if r.get('em') is not None]
    print(f"ảnh thật: {len(rr)} câu, EM {sum(r['em'] for r in g)}/{len(g)}")
D = json.load(open(f'{DEMO}/demo_examples.json', encoding='utf-8'))
print('demo:', [(d['slot'], d.get('image'), d.get('prediction'), d.get('level')) for d in D][:12])

## 8. Mở giao diện 30 phút để quay video demo

Cell in ra dòng **`Running on public URL: https://….gradio.live`**. Mở link đó trên **máy tính**, rồi quay màn hình theo `docs/W14/Demo_Script.md` (mục "Cách quay").

- Ô **Ví dụ** ở dưới khung câu hỏi chính là các ví dụ của mục 6b: bấm vào là ảnh + câu hỏi được điền sẵn.
- Link công khai: ai có link đều dùng được trong lúc cell chạy → **không chia sẻ link**, và **che thanh địa chỉ** khi quay. Giao diện không lưu ảnh / câu hỏi; chỉ ghi số đo thời gian vào `ui_log.jsonl`.
- Sau `UI_MINUTES` phút (30), giao diện tự tắt và notebook chạy tiếp tới bước trả máy. Muốn tắt sớm: bấm ■ (Stop) ở cell này rồi chạy tay cell mục 9.

*Kỳ vọng: `Khởi động: {…}` rồi `Running on public URL: https://….gradio.live`.*

In [ ]:
if UI_MINUTES > 0:
    print(f'mở giao diện {UI_MINUTES} phút | ví dụ: {DEMO}/examples.json')
    !timeout -s INT {UI_MINUTES}m python -u -m src.prototype.app --adapter "{AD_SH}" --expect_adapter_sha {AD_SHA} --ocr_device {OCR_DEVICE} --ocr_log {OUT}/ui_ocr_worker.log --examples_json {DEMO}/examples.json --log_jsonl {OUT}/ui_log.jsonl --share
    print('Giao diện đã tắt.')
else:
    print('UI_MINUTES = 0 -> không mở giao diện')

## 9. Tự trả máy L4

Kết quả đã nằm trên Drive. Nếu notebook dừng sớm vì lỗi thì máy **không** tự trả: đọc lỗi xong, tự bấm **Runtime → Disconnect and delete runtime**.

*Kỳ vọng: `✅ Xong toàn bộ -> tự trả máy …`.*

In [ ]:
AUTO_RELEASE = True
if AUTO_RELEASE:
    print('✅ Xong toàn bộ -> tự trả máy để ngừng tính CU (output phía trên vẫn giữ trong tab).')
    from google.colab import runtime
    runtime.unassign()
else:
    print('AUTO_RELEASE = False -> nhớ tự Disconnect and delete runtime khi xong.')